# Collaborative Hackathon: Classification Challenge — Loan Approval

**Activity 4.2**
Kevin Curtis, Business Systems Manager, Xtrac Limited

---

This notebook works through the six stages set out in the hackathon brief, using the
Loan Approval dataset. The brief offered four datasets, two for regression and two for
classification, and this one was selected.

| Brief stage | Section |
|---|---|
| 1. Define the problem | 1 |
| 2. Choose the ML approach | 2 |
| 3. Data preparation | 3 |
| 4. Model selection and training | 4 |
| 5. Hyperparameter tuning | 5 |
| 6. Model evaluation | 6 |
| Deployment prototype | 7 |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score,
                             precision_recall_curve, ConfusionMatrixDisplay)

RANDOM_SEED = 42
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

print("libraries loaded")

---
## 1. Define the problem

**Business context.** A lending institution decides whether to approve or reject loan
applications. Each decision carries a cost in both directions. Approving an applicant who
later defaults loses the outstanding balance. Rejecting an applicant who would have repaid
loses the interest income and, over time, the customer.

**The current difficulty.** Applications are assessed against criteria applied by
individual underwriters. The same application can receive different outcomes depending on
who reviews it, and the volume of applications makes consistent manual review expensive.

**How an ML solution would address it.** Historical applications and their outcomes are
already recorded. A supervised model trained on that history applies the same criteria to
every application, produces a probability rather than a binary opinion, and makes the
basis of its decisions inspectable. It supports the underwriter rather than replacing
them: the decision threshold remains a business choice.

**Goal.** Predict whether a loan application will be approved.

In [ ]:
applications = pd.read_csv("loan_data.csv")

print("rows and columns:", applications.shape)
applications.head()

### Column meanings

| Column | Meaning | Type |
|---|---|---|
| `person_age` | Applicant age | Numeric |
| `person_gender` | Gender | Categorical |
| `person_education` | Highest education level | Categorical |
| `person_income` | Annual income | Numeric |
| `person_emp_exp` | Years of employment experience | Numeric |
| `person_home_ownership` | Home ownership status | Categorical |
| `loan_amnt` | Amount requested | Numeric |
| `loan_intent` | Purpose of the loan | Categorical |
| `loan_int_rate` | Interest rate offered | Numeric |
| `loan_percent_income` | Loan as a proportion of annual income | Numeric |
| `cb_person_cred_hist_length` | Length of credit history in years | Numeric |
| `credit_score` | Credit score | Numeric |
| `previous_loan_defaults_on_file` | Previous defaults recorded | Categorical |
| `loan_status` | **Target.** 1 = approved, 0 = rejected | Numeric |

The dataset is synthetic, derived from the Kaggle Credit Risk dataset and enlarged using
SMOTENC. This is noted here because it affects how far the results can be generalised,
and it is returned to in section 6.

In [ ]:
print(applications.dtypes.to_string())
print("\nmissing values:", applications.isna().sum().sum())
print("\napproval rate:", applications["loan_status"].mean().round(4))
print(applications["loan_status"].value_counts().to_string())

In [ ]:
figure, axis = plt.subplots(figsize=(5, 4))
applications["loan_status"].value_counts().plot(kind="bar", ax=axis, rot=0)
axis.set_xticklabels(["Rejected (0)", "Approved (1)"])
axis.set_ylabel("applications")
axis.set_title("Class distribution")
plt.tight_layout()
plt.show()

The classes are imbalanced at roughly 22% approved to 78% rejected. This is not severe,
but it is enough that accuracy alone would be misleading: a model rejecting every
application would be 78% accurate. Metric choice is addressed in section 2.

---
## 2. Choose the machine learning approach

**Problem type.** Supervised binary classification. The target is a two-valued category
and labelled historical outcomes are available.

**Target variable.** `loan_status`, where 1 indicates approval.

**Independent variables.** Eight numeric and five categorical columns describing the
applicant and the requested loan.

**Algorithms selected.**

- *Logistic regression*, as an interpretable baseline. It produces coefficients that can
  be read as the effect of each feature on the odds of approval, which matters in a
  lending context where decisions may need explaining to an applicant or a regulator.
- *Random forest*, as a flexible non-linear comparison. It captures interactions between
  features without those interactions being specified in advance, at the cost of
  individual decisions being harder to explain.

**Evaluation metrics.** Accuracy is reported for completeness but not relied upon. The
primary metrics are precision, recall, F1 and ROC-AUC, with precision-recall AUC also
recorded because the positive class is the minority. Precision and recall correspond
directly to the two business costs: precision reflects how many approved applicants were
genuinely creditworthy, recall reflects how many creditworthy applicants were captured.

---
## 3. Data preparation

### 3.1 Missing values

The dataset contains no missing values, which is consistent with it being synthetically
generated. This is confirmed below rather than assumed.

In [ ]:
missing = applications.isna().sum()
print(missing[missing > 0] if missing.sum() else "No missing values in any column.")

### 3.2 Outliers and data quality

Summary statistics reveal values that are not plausible rather than merely extreme.

In [ ]:
print(applications[["person_age", "person_emp_exp", "person_income"]].describe().round(2).to_string())

print("\nimplausible values:")
print("  age over 100:", (applications.person_age > 100).sum())
print("  age over 80:", (applications.person_age > 80).sum())
print("  employment experience over 60 years:", (applications.person_emp_exp > 60).sum())
print("  income over 1,000,000:", (applications.person_income > 1_000_000).sum())

A maximum age of 144 and 125 years of employment experience are not outliers in the
statistical sense, they are data errors. They are removed rather than capped, because a
record containing an impossible value cannot be trusted in its other fields either.

High incomes are treated differently. An income of over £1,000,000 is unusual but not
impossible, so those records are retained. Removing genuine extreme values would bias the
model against high earners.

In [ ]:
before = len(applications)

clean = applications[
    (applications.person_age <= 80) &
    (applications.person_emp_exp <= 60)
].copy()

print(f"removed {before - len(clean)} records with implausible values "
      f"({(before - len(clean)) / before:.3%} of the dataset)")
print("remaining:", len(clean))
print("approval rate after cleaning:", clean.loan_status.mean().round(4))

### 3.3 Feature selection

Cross-tabulating the target against `previous_loan_defaults_on_file` produces a result
worth examining before any model is trained.

In [ ]:
print(clean.groupby("previous_loan_defaults_on_file")["loan_status"]
      .agg(["count", "mean"]).round(4).to_string())

**Every applicant with a previous default on file was rejected**, across roughly 22,800
records, with an approval rate of exactly zero.

This is not a pattern the model would discover. It is a deterministic business rule
already present in the data. A model trained with this column included would learn the
rule and rely on it almost entirely, producing high scores while adding nothing to a
one-line condition that already exists in the lending policy.

The column is therefore excluded from the feature set. The effect of that decision is
measured in section 4.4 rather than assumed, and the exclusion is acknowledged in the
evaluation: the model's scope is applications from applicants with no previous default,
which is the population where a decision is genuinely required.

`person_gender` is also excluded. Using gender as a lending criterion would be
discriminatory, and its inclusion could not be justified regardless of any predictive
value it carried.

In [ ]:
target_column = "loan_status"
excluded = ["previous_loan_defaults_on_file", "person_gender"]

numeric_features = ["person_age", "person_income", "person_emp_exp", "loan_amnt",
                    "loan_int_rate", "loan_percent_income",
                    "cb_person_cred_hist_length", "credit_score"]
categorical_features = ["person_education", "person_home_ownership", "loan_intent"]

input_columns = numeric_features + categorical_features

print(f"features used: {len(input_columns)}")
for c in input_columns:
    print("  ", c)
print("\nexcluded:", excluded)

### 3.4 Partitioning the data

The data is split 70% training, 15% validation, 15% test.

The reasoning behind these ratios is as follows. The validation set is used for model
comparison and hyperparameter selection, and the test set is held back entirely until
section 6 so that the final figures are produced on data that influenced no decision. With
approximately 45,000 records, 15% gives around 6,700 rows per holdout set, which contains
roughly 1,500 approvals. That is sufficient for stable precision and recall estimates
while leaving the majority of the data available for training.

A larger training proportion would improve the model marginally but would make the holdout
metrics noisier. A smaller one would waste data on evaluation that is already
sufficiently precise.

Both splits are stratified on the target so that the approval rate is preserved in all
three sets.

In [ ]:
inputs = clean[input_columns]
target = clean[target_column]

# first split: hold out 15% as the test set
train_val_inputs, test_inputs, train_val_target, test_target = train_test_split(
    inputs, target, test_size=0.15, random_state=RANDOM_SEED, stratify=target)

# second split: 15% of the original becomes validation (0.1765 of what remains)
train_inputs, validation_inputs, train_target, validation_target = train_test_split(
    train_val_inputs, train_val_target, test_size=0.1765,
    random_state=RANDOM_SEED, stratify=train_val_target)

for name, t in [("training", train_target), ("validation", validation_target),
                ("test", test_target)]:
    print(f"{name:11s} {len(t):6d} rows   approval rate {t.mean():.4f}   approvals {t.sum()}")

### 3.5 Encoding and standardisation

Categorical columns are one-hot encoded. Assigning numbers to categories such as home
ownership would imply an ordering and a spacing between them that does not exist.

Numeric columns are standardised so that each is expressed as deviations from its own
mean. Income is measured in tens of thousands and interest rate in single digits, and
without scaling the regularisation penalty in logistic regression would fall unevenly
across features purely because of their units. Tree-based models are unaffected by
monotonic rescaling, but applying the same preparation to both keeps the comparison fair.

Both steps are placed inside a `Pipeline`. This matters for more than tidiness: the scaler
learns its mean and standard deviation from the training rows only and reuses them on
validation and test. Fitting the scaler on the full dataset would leak information from
the holdout sets into training and inflate the reported scores.

In [ ]:
def make_pipeline(algorithm):
    preparation = ColumnTransformer([
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         categorical_features),
    ])
    return Pipeline([("preparation", preparation), ("model", clone(algorithm))])

print("pipeline defined")

---
## 4. Model selection and training

Both models are trained on the training set and compared on the validation set. The test
set is not used at this stage.

In [ ]:
def evaluate(fitted_pipeline, evaluation_inputs, evaluation_target, label):
    probability = fitted_pipeline.predict_proba(evaluation_inputs)[:, 1]
    prediction = (probability >= 0.5).astype(int)
    return {
        "model": label,
        "accuracy": (prediction == evaluation_target).mean(),
        "precision": precision_score(evaluation_target, prediction, zero_division=0),
        "recall": recall_score(evaluation_target, prediction),
        "f1": f1_score(evaluation_target, prediction),
        "roc_auc": roc_auc_score(evaluation_target, probability),
        "pr_auc": average_precision_score(evaluation_target, probability),
    }


algorithms = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_SEED),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED,
                                            n_jobs=-1),
}

validation_results = []
fitted = {}
for label, algorithm in algorithms.items():
    pipeline = make_pipeline(algorithm)
    pipeline.fit(train_inputs, train_target)
    fitted[label] = pipeline
    validation_results.append(evaluate(pipeline, validation_inputs,
                                       validation_target, label))

# baseline for comparison: reject every application
baseline_prediction = np.zeros(len(validation_target))
validation_results.insert(0, {
    "model": "Baseline (reject all)",
    "accuracy": (baseline_prediction == validation_target).mean(),
    "precision": 0.0, "recall": 0.0, "f1": 0.0, "roc_auc": 0.5, "pr_auc": validation_target.mean(),
})

validation_table = pd.DataFrame(validation_results).round(4)
validation_table

### 4.1 Cross-validated comparison

The comparison above rests on a single validation slice. If that slice happened to fall
favourably for one model, the conclusion would be unreliable.

Five-fold stratified cross-validation addresses this. The training data is divided into
five parts; each model is trained five times, holding out a different part each time, and
the five scores are averaged. The standard deviation across folds indicates how stable the
result is. A large gap between models combined with a small spread across folds gives
confidence that the difference is real rather than an artefact of one partition.

The test and validation sets play no part in this. Cross-validation runs entirely within
the training data.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

cross_validated = []
for label, algorithm in algorithms.items():
    auc_scores = cross_val_score(make_pipeline(algorithm), train_inputs, train_target,
                                 cv=folds, scoring="roc_auc", n_jobs=-1)
    f1_scores = cross_val_score(make_pipeline(algorithm), train_inputs, train_target,
                                cv=folds, scoring="f1", n_jobs=-1)
    cross_validated.append({
        "model": label,
        "roc_auc_mean": auc_scores.mean(), "roc_auc_std": auc_scores.std(),
        "roc_auc_min": auc_scores.min(), "roc_auc_max": auc_scores.max(),
        "f1_mean": f1_scores.mean(), "f1_std": f1_scores.std(),
    })
    print(f"{label:22s} fold ROC-AUCs: {auc_scores.round(4)}")

cross_validation_table = pd.DataFrame(cross_validated).round(4)
cross_validation_table

In [ ]:
# Does the better model win on every fold, or only on average?
fold_scores = {}
for label, algorithm in algorithms.items():
    fold_scores[label] = cross_val_score(make_pipeline(algorithm), train_inputs,
                                         train_target, cv=folds,
                                         scoring="roc_auc", n_jobs=-1)

fold_table = pd.DataFrame(fold_scores, index=[f"fold {i+1}" for i in range(5)]).round(4)
fold_table["difference"] = (fold_table["Random Forest"]
                            - fold_table["Logistic Regression"]).round(4)
print(fold_table.to_string())

wins = (fold_table["difference"] > 0).sum()
print(f"\nRandom Forest ahead on {wins} of 5 folds")
print(f"smallest margin: {fold_table['difference'].min():.4f}")
print(f"largest fold-to-fold variation within a model: "
      f"{max(fold_table['Random Forest'].std(), fold_table['Logistic Regression'].std()):.4f}")

In [ ]:
figure, axis = plt.subplots(figsize=(7, 4))
axis.boxplot([fold_scores["Logistic Regression"], fold_scores["Random Forest"]],
             tick_labels=["Logistic Regression", "Random Forest"])
for position, label in enumerate(["Logistic Regression", "Random Forest"], start=1):
    axis.scatter([position] * 5, fold_scores[label], alpha=0.6, zorder=3)
axis.set_ylabel("ROC-AUC")
axis.set_title("Cross-validated performance across 5 folds")
plt.tight_layout()
plt.show()

> **Read the spread, not just the means.** If the margin between the two models on every
> fold is larger than the variation within either model across folds, the difference is
> not attributable to how the data was partitioned. If the boxes overlap, the single-split
> comparison in the previous section overstated the difference.

### 4.2 Comparison and model choice

Interpret the table above rather than simply reporting it. Points to consider:

- How far each model beats the reject-all baseline, and on which metrics.
- Whether the two models differ mainly in precision or in recall, and what that means for
  the lending decision.
- Whether the random forest's advantage, if any, is large enough to justify the loss of
  interpretability relative to logistic regression.

In [ ]:
figure, axis = plt.subplots(figsize=(8, 4))
plot_table = validation_table.set_index("model")[["precision", "recall", "f1", "roc_auc"]]
plot_table.plot(kind="bar", ax=axis, rot=15)
axis.set_ylabel("score")
axis.set_title("Validation performance")
axis.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

### 4.3 What the models rely on

The coefficients of the logistic regression indicate both the size and the direction of
each feature's effect. Positive values increase the odds of approval.

In [ ]:
logistic_pipeline = fitted["Logistic Regression"]
encoded_names = list(logistic_pipeline.named_steps["preparation"]
                     .named_transformers_["categorical"]
                     .get_feature_names_out(categorical_features))
feature_names = numeric_features + encoded_names

coefficients = pd.Series(logistic_pipeline.named_steps["model"].coef_[0],
                         index=feature_names).sort_values()

figure, axis = plt.subplots(figsize=(7, 6))
coefficients.plot(kind="barh", ax=axis)
axis.axvline(0, c="black", lw=0.8)
axis.set_xlabel("coefficient (log-odds)")
axis.set_title("Logistic regression coefficients")
plt.tight_layout()
plt.show()

print(coefficients.round(3).to_string())

### 4.4 Measuring the effect of the excluded column

Section 3.3 excluded `previous_loan_defaults_on_file` on the grounds that it encodes an
existing business rule. That decision is tested here rather than asserted, by training the
same model with the column restored.

In [ ]:
with_default_flag = numeric_features + categorical_features + ["previous_loan_defaults_on_file"]

leak_train, leak_test, leak_train_y, leak_test_y = train_test_split(
    clean[with_default_flag], clean[target_column],
    test_size=0.15, random_state=RANDOM_SEED, stratify=clean[target_column])

leak_preparation = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
     categorical_features + ["previous_loan_defaults_on_file"]),
])
leak_pipeline = Pipeline([("preparation", leak_preparation),
                          ("model", clone(algorithms["Random Forest"]))])
leak_pipeline.fit(leak_train, leak_train_y)

leak_probability = leak_pipeline.predict_proba(leak_test)[:, 1]
leak_prediction = (leak_probability >= 0.5).astype(int)

print("Random forest WITH previous_loan_defaults_on_file included")
print(f"  precision {precision_score(leak_test_y, leak_prediction, zero_division=0):.4f}")
print(f"  recall    {recall_score(leak_test_y, leak_prediction):.4f}")
print(f"  f1        {f1_score(leak_test_y, leak_prediction):.4f}")
print(f"  roc_auc   {roc_auc_score(leak_test_y, leak_probability):.4f}")

importances = pd.Series(
    leak_pipeline.named_steps["model"].feature_importances_,
    index=numeric_features + list(leak_pipeline.named_steps["preparation"]
          .named_transformers_["categorical"]
          .get_feature_names_out(categorical_features + ["previous_loan_defaults_on_file"])))
print("\ntop 6 features by importance:")
print(importances.sort_values(ascending=False).head(6).round(4).to_string())

> **Record the comparison.** Note how much the scores improve with the column included,
> and where the default flag ranks among the feature importances. If it dominates, that
> confirms the model is reproducing the existing rule rather than learning anything
> additional, and supports the decision to exclude it.

---
## 5. Hyperparameter tuning

Grid search with three-fold cross-validation is applied to the better performing model
from section 4. Cross-validation is used in preference to the single validation split so
that the selected parameters are not tuned to one particular partition of the data.

The parameter ranges tested are recorded below. They were chosen to span the main
trade-off in a random forest: `max_depth` and `min_samples_split` control how far each
tree is allowed to fit the training data, while `n_estimators` controls how many trees are
averaged. Deeper trees reduce bias and increase variance; more trees reduce variance at a
cost in training time.

Scoring uses ROC-AUC rather than accuracy, for the reasons given in section 2.

In [ ]:
parameter_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20, None],
    "model__min_samples_split": [2, 10],
}

print("combinations to test:",
      np.prod([len(v) for v in parameter_grid.values()]), "x 3 folds")

grid_search = GridSearchCV(
    make_pipeline(algorithms["Random Forest"]),
    parameter_grid,
    cv=3,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(train_inputs, train_target)

print("\nbest parameters:", grid_search.best_params_)
print(f"best cross-validated ROC-AUC: {grid_search.best_score_:.4f}")

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
summary = results[["param_model__n_estimators", "param_model__max_depth",
                   "param_model__min_samples_split",
                   "mean_test_score", "std_test_score", "rank_test_score"]]
summary.columns = ["n_estimators", "max_depth", "min_samples_split",
                   "mean_roc_auc", "std", "rank"]
summary.sort_values("rank").round(4)

> **Interpret the grid.** Comment on how much difference the tuning actually made. If the
> spread between the best and worst combinations is small relative to the standard
> deviation across folds, the model is not sensitive to these parameters and tuning has
> bought little. That is a legitimate finding and is worth stating, because reporting a
> tuned model implies the tuning mattered.

In [ ]:
tuned_model = grid_search.best_estimator_

tuning_comparison = pd.DataFrame([
    evaluate(fitted["Random Forest"], validation_inputs, validation_target, "Random Forest (default)"),
    evaluate(tuned_model, validation_inputs, validation_target, "Random Forest (tuned)"),
]).round(4)
tuning_comparison

---
## 6. Model evaluation

The tuned model is evaluated on the test set, which has not been used at any point in
model selection or tuning. These are the figures that should be reported.

In [ ]:
test_probability = tuned_model.predict_proba(test_inputs)[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)

print(f"test set: {len(test_target)} applications, {test_target.sum()} approvals\n")
print(classification_report(test_target, test_prediction,
                            target_names=["Rejected", "Approved"], digits=4))
print(f"ROC-AUC: {roc_auc_score(test_target, test_probability):.4f}")
print(f"PR-AUC:  {average_precision_score(test_target, test_probability):.4f}")

In [ ]:
matrix = confusion_matrix(test_target, test_prediction)
print(pd.DataFrame(matrix,
                   index=["actually rejected", "actually approved"],
                   columns=["predicted reject", "predicted approve"]).to_string())

figure, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay(matrix, display_labels=["Rejected", "Approved"]).plot(
    ax=axes[0], colorbar=False)
axes[0].set_title("Confusion matrix (test set)")

precision_points, recall_points, _ = precision_recall_curve(test_target, test_probability)
axes[1].plot(recall_points, precision_points)
axes[1].axhline(test_target.mean(), ls="--", c="grey", label="no-skill baseline")
axes[1].set_xlabel("recall")
axes[1].set_ylabel("precision")
axes[1].set_title("Precision-recall curve")
axes[1].legend()
plt.tight_layout()
plt.show()

### 6.1 What the metrics indicate

Work through each in turn:

- **Precision** — of the applications the model would approve, the proportion that were
  genuinely approvals. This is the metric attached to the cost of a bad loan.
- **Recall** — of the applications that should have been approved, the proportion the
  model captured. This is the metric attached to lost business.
- **F1** — the balance between them, useful for ranking models but not itself a business
  target.
- **ROC-AUC** — performance across all thresholds. Note that with a 22% positive class it
  is somewhat flattered by the large number of true negatives.
- **PR-AUC** — the same idea but ignoring true negatives, so a more conservative reading.

The confusion matrix converts these into counts, which is usually the more useful form for
a non-technical audience: a specific number of loans wrongly approved and a specific
number of good applicants wrongly turned away.

### 6.2 Threshold as a business decision

The 0.5 cut-off is a library default rather than a choice derived from the problem. Moving
it trades the two error types against one another without retraining the model.

In [ ]:
sweep = []
for threshold in np.arange(0.1, 0.91, 0.05):
    prediction = (test_probability >= threshold).astype(int)
    sweep.append({
        "threshold": round(threshold, 2),
        "precision": precision_score(test_target, prediction, zero_division=0),
        "recall": recall_score(test_target, prediction),
        "f1": f1_score(test_target, prediction, zero_division=0),
        "bad_approvals": int(((prediction == 1) & (test_target == 0)).sum()),
        "missed_good": int(((prediction == 0) & (test_target == 1)).sum()),
    })

threshold_table = pd.DataFrame(sweep).round(4)
threshold_table

### 6.3 Limitations

- The dataset is synthetic, generated from the Kaggle Credit Risk data and enlarged using
  SMOTENC. Synthetic records reproduce the statistical structure of the original but
  contain no genuinely new information, which places a ceiling on how far the results
  generalise to real lending.
- The model's scope is limited by the exclusion in section 3.3. It predicts approval for
  applicants with no previous default on file, because for applicants with one the outcome
  is already determined by policy.
- `loan_status` records the decision that was made, not whether the loan was subsequently
  repaid. The model therefore predicts the behaviour of the existing approval process,
  including any inconsistency or bias within it, rather than creditworthiness itself.
- Approximately 0.04% of records were removed as data errors. Their removal is unlikely to
  affect the result but is recorded for completeness.

---
## 7. Deployment prototype

The final model is saved and wrapped in a minimal Flask API, following the pattern in the
hackathon template. The whole pipeline is saved, not just the fitted algorithm, so that
incoming applications receive the same encoding and scaling that the training data
received. Saving the model alone is a common cause of predictions that are correct in
development and wrong in production.

In [ ]:
import joblib

joblib.dump(tuned_model, "best_model.pkl")
print("saved best_model.pkl")

example_application = pd.DataFrame([{
    "person_age": 32, "person_income": 54000, "person_emp_exp": 8,
    "loan_amnt": 9000, "loan_int_rate": 11.2, "loan_percent_income": 0.17,
    "cb_person_cred_hist_length": 6, "credit_score": 655,
    "person_education": "Bachelor", "person_home_ownership": "RENT",
    "loan_intent": "PERSONAL",
}])

print("\nexample application:")
print(f"  approval probability: {tuned_model.predict_proba(example_application)[0, 1]:.4f}")
print(f"  decision at 0.5: {'APPROVE' if tuned_model.predict(example_application)[0] else 'REJECT'}")

In [ ]:
%%writefile app.py
from flask import Flask, request, jsonify
import joblib
import pandas as pd

app = Flask(__name__)
model = joblib.load("best_model.pkl")

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()
    input_df = pd.DataFrame([data])
    probability = float(model.predict_proba(input_df)[0, 1])
    return jsonify({
        "approval_probability": probability,
        "approved": probability >= 0.5,
    })

if __name__ == "__main__":
    app.run(debug=True)

The endpoint returns the probability alongside the decision. This is deliberate: it
allows the threshold to be set by the business rather than fixed in the model, and it
gives an underwriter reviewing a borderline case the information needed to exercise
judgement.

---

## Reference

Loan Approval Classification Dataset. Synthetic dataset derived from the Kaggle Credit
Risk dataset, enlarged using SMOTENC. Supplied as part of Activity 4.2.